# Учительская проверка submission: линейная регрессия

Этот блокнот не нужно раздавать участникам вместе с ответами. Он читает файлы команд из папки `submissions` и сравнивает их со скрытым `data/private_target.csv`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = Path("data")
SUBMISSIONS_DIR = Path("submissions")
TARGET = "price_mln"

private = pd.read_csv(DATA_DIR / "private_target.csv")
files = sorted(SUBMISSIONS_DIR.glob("*.csv"))

rows = []
for path in files:
    sub = pd.read_csv(path)
    merged = private.merge(sub, on="id", how="inner", suffixes=("_true", "_pred"))
    if len(merged) != len(private):
        print(f"Пропускаю {path.name}: найдено {len(merged)} id из {len(private)}")
        continue
    y_true = merged[f"{TARGET}_true"]
    y_pred = merged[f"{TARGET}_pred"]
    rows.append(
        {
            "team_file": path.name,
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred),
        }
    )

if rows:
    leaderboard = pd.DataFrame(rows).sort_values(["MAE", "RMSE"], ascending=[True, True])
    display(leaderboard)
else:
    print("Положите CSV-файлы команд в папку submissions и запустите ячейку снова.")
